In [45]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv

In [46]:
load_dotenv()

True

In [47]:
llm = ChatGroq(
    model = "openai/gpt-oss-20b",
    temperature = 0.4
)

In [35]:
class EProduct(TypedDict):
    order_amount : float
    discount : float
    delievery_charge : int
    item_purchased : int
    customer_rating : float

    ##state to be returned##
# 
    final_price : float
    discount_percentage : float
    avg_item_cost : float
    customer_feedback : str
    order_category : str
    summary : str

In [48]:
def final_price_func(state : EProduct):
    amount = state["order_amount"]
    discount = state["discount"]
    delievery_charge = state["delievery_charge"]
    
    final_price = amount-discount+delievery_charge

    return {
        "final_price" : final_price
    }

In [49]:
def discount_func(state : EProduct):
    amount = state["order_amount"]
    discount = state["discount"]

    discount_percentage = (discount/amount)*100

    return {
        "discount_percentage" : discount_percentage
    }

In [50]:
def avg_item_func(state : EProduct):
    amount = state["order_amount"]
    item_purchased = state["item_purchased"]
    
    avg_item_cost = amount/item_purchased

    return {
        "avg_item_cost" : avg_item_cost
    }

In [51]:
def customer_feedback_func(state : EProduct):
    rating = state["customer_rating"]

    if rating >= 4:
        return {"customer_feedback" : "excellent"}
    elif rating >= 3 and rating <= 3.9:
        return {"customer_feedback" : "average"}
    else:
        return {"customer_feedback" : "poor"}
    


In [52]:
def order_category_func(state : EProduct):
    amount = state["order_amount"]
    if amount >= 4000:
        return {"order_category" : "premium"}
    elif amount >= 1500 and amount < 4000:
        return {"order_category" : "standard"}
    else:
        return {"order_category" : "basic"}

In [54]:
def summary_func(state: EProduct):

    prompt = f"""
    You are an e-commerce order analyst.

    Analyze the following order details:

    Final Price: ₹{state['final_price']}
    Discount Percentage: {state['discount_percentage']}%
    Average Item Cost: ₹{state['avg_item_cost']}
    Customer Feedback: {state['customer_feedback']}
    Order Category: {state['order_category']}

    Generate a concise order analysis in 3-4 sentences.
    Mention the final price, discount benefit, customer
    satisfaction, and order category.

    Keep the language professional and easy to understand.
    """

    answer = llm.invoke(prompt)

    return {"summary": answer.content}

In [55]:
graph = StateGraph(EProduct)

graph.add_node("final_price_func", final_price_func)
graph.add_node("discount_func", discount_func)
graph.add_node("avg_item_func", avg_item_func)
graph.add_node("customer_feedback_func", customer_feedback_func)
graph.add_node("order_category_func", order_category_func)
graph.add_node("summary", summary_func)

graph.add_edge(START, "final_price_func")
graph.add_edge(START, "discount_func")
graph.add_edge(START, "avg_item_func")
graph.add_edge(START, "customer_feedback_func")
graph.add_edge(START, "order_category_func")

graph.add_edge("final_price_func", "summary")
graph.add_edge("discount_func", "summary")
graph.add_edge("avg_item_func", "summary")
graph.add_edge("customer_feedback_func", "summary")
graph.add_edge("order_category_func", "summary")

workflow = graph.compile()


In [58]:
initial_state = {
    "order_amount": 500,
    "discount": 50,
    "delievery_charge": 10,
    "item_purchased": 2,
    "customer_rating": 1.5
}
final_state = workflow.invoke(initial_state)
print(final_state["summary"])

The order was completed at a final price of ₹460 after applying a 10 % discount, which reduced the original cost by ₹46.  
The discount benefit helped keep the purchase within a reasonable budget, especially given the average item cost of ₹250.  
Customer feedback was marked as “poor,” indicating dissatisfaction that may need to be addressed in future interactions.  
This transaction falls under the “basic” order category, reflecting a standard, low‑volume purchase.
